# DeepFace-PAD · OULU-NPU Protocol 1 trên Colab
Chạy cell 1 và 2, rồi cell 4 và 5. Cell 3 là pilot tùy chọn. Chọn runtime GPU A100 và bấm **Connect**. Cell đầu xác nhận GPU, VRAM và thư mục Drive. Nếu bộ dữ liệu ở Shared with me, thêm shortcut vào My Drive và điền đúng đường dẫn vào `DATA`.

Lượt nhanh dùng MobileNetV3-Small, 8 khung hình/video và 3 epoch. Code và YuNet đã đi kèm notebook. Kết quả ghi vào My Drive. Phần Test ở cell riêng, chạy sau khi đã có checkpoint và ngưỡng khóa từ Dev.

In [ ]:
#@title 1. Kết nối Drive và kiểm tra GPU
from pathlib import Path
from google.colab import drive
import shutil
import subprocess
import sys
import torch

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/OULU-NPU")  # sửa nếu tên shortcut khác
WORK = Path("/content/deepface-pad-oulu")
CACHE = WORK / "cache"
RUNS = Path("/content/drive/MyDrive/face-pad/oulu-fast-p1")
assert torch.cuda.is_available(), "Chọn runtime GPU rồi chạy lại cell này"
precision = "bf16" if torch.cuda.is_bf16_supported() else "fp32"
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
for name, size in {"Train_files.tar": 37753662976, "Dev_files.tar": 28151992320,
                   "Protocols.tar": 417792}.items():
    path = DATA / name
    assert path.is_file() and path.stat().st_size == size, f"Kiểm tra đường dẫn và dung lượng: {path}"
print("precision:", precision, "| disk free (GiB):", round(shutil.disk_usage("/content").free / 2**30, 1))
RUNS.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 2. Nạp code đi kèm notebook
BUNDLE_SHA256 = '5a1ac970c7a1452773a8682079433356cff7cbadff073484fd145d34bb72dfa1'
BUNDLE = (
    '...'
)
import base64
import hashlib
import io
import os
import tarfile

bundle = base64.b64decode(BUNDLE)
assert hashlib.sha256(bundle).hexdigest() == BUNDLE_SHA256
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        destination = (WORK / member.name).resolve()
        assert member.isfile() and destination.is_relative_to(WORK)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.extractfile(member).read())
import time
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import run_logged
from deepface_pad.colab_cache import prepare_oulu_split
ENV = {**os.environ, "PYTHONPATH": str(WORK / "src")}
def call(module, *args):
    label = "-".join([module.rsplit(".", 1)[-1], str(args[0])])
    run_logged([sys.executable, "-u", "-m", module, *args], label=label,
               log=RUNS / "logs" / f"{label}-{time.time_ns()}.log", env=ENV)

FRAMES = 8
def prepare(split, pilot=False, backup=True):
    return prepare_oulu_split(DATA, WORK, RUNS / "prepared-cache", split, FRAMES, pilot, call, backup=backup)

with open(RUNS / f"source-{BUNDLE_SHA256[:12]}.tar.gz", "wb") as output:
    output.write(bundle)
import cv2
import torchvision
import yaml
print("code sha256", BUNDLE_SHA256, "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| OpenCV", cv2.__version__)

In [ ]:
#@title 3. Pilot tùy chọn, 30 video mỗi split
prepare("train", pilot=True)
prepare("val", pilot=True)
print("Mỗi split pilot cần có 30 video / 240 khung. Xem crop_counts và seconds trong log.")

In [ ]:
#@title 4. Khôi phục hoặc chuẩn bị Train/Dev và lưu lên Drive
BACKUP_RGB = True  # đặt False để train ngay; chạy lại cell này để lưu sau

for split in ("train", "val"):
    manifest = prepare(split, backup=BACKUP_RGB)
    summary = manifest.with_suffix(".json")
    shutil.copyfile(summary, RUNS / f"prep-{split}-{BUNDLE_SHA256[:12]}.json")
print("Mốc kiểm: Train 1.200 / 9.600; Dev 900 / 7.200.")

In [ ]:
#@title 5. Đo batch 64/128, rồi train và khóa ngưỡng từ Dev
# 0 = tự đo batch 64/128. Thời gian mỗi epoch được in ra khi chạy.
BATCH = 0
WORKERS = 2
EPOCHS = 3
SEED = 43
call("deepface_pad.oulu_fast", "train", "--root", CACHE, "--directory", CACHE / "p1",
     "--runs", RUNS, "--batch", BATCH, "--workers", WORKERS, "--epochs", EPOCHS,
     "--seed", SEED, "--precision", precision)

In [ ]:
#@title 6. Chấm Test một lần với checkpoint đã khóa
# Điền đúng đường dẫn từ RUN_DIR= ở cuối cell 5.
RUN_DIR = ""
assert RUN_DIR, "Điền RUN_DIR từ cell 5"
run_dir = Path(RUN_DIR)
assert (run_dir / "freeze.json").is_file(), "Run chưa khóa"
assert not (run_dir / "test_metrics.json").exists(), "Run này đã có kết quả Test"
prepare("test")
shutil.copyfile(CACHE / "p1/test.json", run_dir / "test_preparation.json")
call("deepface_pad.oulu_fast", "score", "--root", CACHE, "--manifest", CACHE / "p1/test.csv",
     "--output", run_dir)

Nếu có lỗi, gửi nguyên traceback và dòng ngay trước đó. Nếu chạy xong, gửi `runtime.json`, `train_log.csv`, `metrics.json`, `test_metrics.json` và `throughput.json` (ở thư mục setup của lượt chạy). APCER và BPCER trong JSON là tỷ lệ 0–1. Nhân 100 để đọc phần trăm.